## Cloning the repository - Google Colab

This step is needed for importing the src scripts. This is needed when execution through google colab

In [ ]:
!pip install -U ipython

In [ ]:
%load_ext autoreload
%autoreload 2

#Clone the repository
import os
if not os.path.exists("your-project"):
    !git clone https://github.com/ras112git/predictive_modeling_and_mobility_forecasting.git
else:
    print("Repo already cloned.")


%cd predictive_modeling_and_mobility_forecasting

# Install dependencies
!pip install -r requirements.txt

import sys
sys.path.append(os.getcwd())

## Setting up in the virtual environment

This cell need to be run, when the data will be cleaned localy.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


## Data cleaning process

This section summarizes all the data cleaning steps taken in the dataset, before starting with the training.

In [ ]:
from src.download_data import download_raw_data

train_path, test_path = download_raw_data()

# Import the raw datasets
import pandas as pd

df_train = pd.read_csv(train_path)
# df_test = pd.read_csv(test_path)

# and also saving the data IDs
train_ids = df_train.iloc[:, 0].astype(str) + '_' + df_train.iloc[:, 1].astype(str)

Raw data already exists at data/raw\dataset_train.csv, skipping download.
Raw data already exists at data/raw\dataset_test.csv, skipping download.


### Data cleaning.
First we need to remove all the Na factors in our dataset. 

In [ ]:
# View the actual NA rows
df_train[df_train.isna().any(axis=1)]

# If there are rows with NA, then drop them
if df_train.isna().any(axis=1).any():
    df_train = df_train.dropna()


#No rows are NA

,datetime,station_number,name,lat,lng,bikes


In [ ]:
#Find duplicates (and show)
df_train[df_train.duplicated()]

# If there are duplicates, then drop them.
if df_train.duplicated().any():
    df_train = df_train.drop_duplicates()

,datetime,station_number,name,lat,lng,bikes


In [ ]:
#Transform the datetime
df_train['datetime'] = pd.to_datetime(df_train['datetime'], utc=True)


In [ ]:
#I want to split my datainto the datapart (how already fastai tabular uses to learn)
from fastai.tabular.all import add_datepart

#drops also the original column
df_train = add_datepart(df_train, 'datetime', time=True, drop=True)

# Cyclical encoding may also be needed, look into this 

In [ ]:
# and now save it to the interim path(maybe we change this later for full path) 
from pathlib import Path

out_path = Path('data/interim/cleaned_data_dates.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df_train.to_csv(out_path, index=False)

In [ ]:
# Now I would need to remove the redundant columns
# Probably I will remove everything except station number.
